# 주택 가격 예측하기

메인퀘스트에 오신 것을 환영합니다!

이번 퀘스트의 목표는 여러 특성들을 종합적으로 고려하여 가장 주택 가격을 잘 예측하는 회귀모델을 만드는 것입니다.

- 예측할 값(y): SalePrice
- 평가지표: RMSE

예시 형식:
~~~
Id,SalePrice
1461,169000.1
1462,187724.1233
1463,175221
etc.
~~~

**최종모델명: 새 상호작용 변수 + 로그 변환 Lasso + 그레이디언트 부스팅 앙상블**

이 노트북은 `train.csv`(1,168채, 정답 포함)로 모델을 만들고 `test.csv`(292채, 정답 없음)의 거래가를 예측해 `submission.csv`를 만든다.

| 단계 | 내용 |
|---|---|
| 1. 데이터 불러오기 | train.csv, test.csv |
| 2. 전처리 | 빈 값 처리, 파생 변수, 품질 등급 점수, **상호작용 변수 7개** |
| 3. 이상치 | 비정상 거래 2채를 학습에서만 제외 |
| 4. 원핫 인코딩 | train·test 열 구성 맞추기 |
| 5. 5-겹 교차검증 | 로그 Lasso, 그레이디언트 부스팅 각각 채점 |
| 6. 앙상블 비율 | 교차검증 예측으로 최적 비율 계산 + 중첩 검증 |
| 6-2. 80:20 검증 | 템플릿처럼 train을 학습 80% · 검증 20%로 나눠 채점 |
| 6-2. 80:20 검증 | 템플릿 분할 코드로 학습 934채 / 검증 234채 |
| 7. 최종 학습·제출 | 학습용 80%로 학습한 모델로 test 예측 → submission.csv |

**평가지표:** RMSE(제곱 평균 제곱근 오차, 달러). 크게 빗나간 예측에 더 큰 벌점을 준다.

**결과 요약:** 템플릿 방식 80:20 검증 RMSE **16,885달러**(R² 0.943, MAPE 7.0%)

## 1. 데이터 불러오기

In [1]:
import time
import warnings
import numpy as np
import pandas as pd

# 추가 라이브러리 부르기
from sklearn.model_selection import KFold                     # 5-겹 교차검증용 분할
from sklearn.linear_model import Lasso                        # L1 정규화 선형회귀
from sklearn.ensemble import GradientBoostingRegressor        # 그레이디언트 부스팅 (트리 앙상블)
from sklearn.preprocessing import StandardScaler              # 표준화 (평균 0, 표준편차 1)
from sklearn.pipeline import make_pipeline                    # 전처리 + 모델을 하나로 묶기
from scipy.optimize import nnls                               # 음수 없는 최소제곱 (앙상블 비율 계산)

warnings.filterwarnings("ignore")

train_df = pd.read_csv("train.csv")    # 학습용: 정답(SalePrice) 포함
test_df = pd.read_csv("test.csv")      # 예측용: 정답 없음
print("train:", train_df.shape, "| test:", test_df.shape)

train: (1168, 81) | test: (292, 80)


## 용어정리

- `Lasso`: 계수의 절댓값 합에 벌점을 주는 선형회귀. 쓸모없는 변수의 계수를 정확히 0으로 만들어 과적합을 막는다.
- `GradientBoostingRegressor`: 작은 결정트리를 수천 개 순서대로 쌓아, 앞 트리가 틀린 부분을 다음 트리가 보완하는 모델이다.
- `nnls`: 여러 모델의 예측을 섞을 때, 음수가 아닌 비율 중 오차를 가장 줄이는 조합을 계산한다.

In [2]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1168 entries, 0 to 1167
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1168 non-null   int64  
 1   MSSubClass     1168 non-null   int64  
 2   MSZoning       1168 non-null   object 
 3   LotFrontage    950 non-null    float64
 4   LotArea        1168 non-null   int64  
 5   Street         1168 non-null   object 
 6   Alley          70 non-null     object 
 7   LotShape       1168 non-null   object 
 8   LandContour    1168 non-null   object 
 9   Utilities      1168 non-null   object 
 10  LotConfig      1168 non-null   object 
 11  LandSlope      1168 non-null   object 
 12  Neighborhood   1168 non-null   object 
 13  Condition1     1168 non-null   object 
 14  Condition2     1168 non-null   object 
 15  BldgType       1168 non-null   object 
 16  HouseStyle     1168 non-null   object 
 17  OverallQual    1168 non-null   int64  
 18  OverallC

In [3]:
na = train_df.isnull().sum()
na = na[na > 0].sort_values(ascending=False)
print(pd.DataFrame({
    "결측치 수": na,
    "비율(%)": (na / len(train_df) * 100).round(1)
}))

              결측치 수  비율(%)
PoolQC         1162   99.5
MiscFeature    1120   95.9
Alley          1098   94.0
Fence           941   80.6
MasVnrType      694   59.4
FireplaceQu     545   46.7
LotFrontage     218   18.7
GarageType       69    5.9
GarageYrBlt      69    5.9
GarageFinish     69    5.9
GarageQual       69    5.9
GarageCond       69    5.9
BsmtExposure     24    2.1
BsmtFinType2     24    2.1
BsmtQual         23    2.0
BsmtCond         23    2.0
BsmtFinType1     23    2.0
MasVnrArea        8    0.7
Electrical        1    0.1


## 2. 전처리

전처리에서 고민해야 할 것은 많습니다.

1. 결측치
2. 이상치

결측치는 처리하지 않으면 학습 자체가 불가할 수 있기 때문에, 최소한의 전처리로 결측치를 처리하겠습니다.

여러분들이라면 어떻게 결측치를 처리하면 좋을지 고민해보세요!

In [4]:
# 범주형(문자) 열 목록 + 숫자로 저장됐지만 실제로는 '주택 유형 코드'인 MSSubClass
CAT_COLS = [c for c in train_df.columns if not pd.api.types.is_numeric_dtype(train_df[c])] + ["MSSubClass"]

# 빈 값이면 '시설 없음'이므로 0으로 채울 숫자 열 (외벽 장식, 지하실, 차고 면적·개수)
ZERO_COLS = ["MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
             "BsmtFullBath", "BsmtHalfBath", "GarageCars", "GarageArea"]

# 품질 등급 글자 → 점수 (Ex 최상 5 ~ Po 나쁨 1, 시설 없음 0)
GRADE = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
GRADE_COLS = ["ExterQual", "ExterCond", "KitchenQual", "BsmtQual", "BsmtCond", "HeatingQC",
              "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]

def prepare(df, lotfrontage_fill):
    """빈 값 처리 + 파생 변수 + 품질 등급 점수. (상호작용 변수는 원핫 인코딩 뒤 add_interactions에서)
    lotfrontage_fill: 도로 접면 길이 빈 값을 채울 값 (train의 중앙값을 넘김)"""
    d = df.copy()

    # ---- 1) 빈 값 처리 ----
    d["MSSubClass"] = d["MSSubClass"].astype(str)              # 코드 숫자 → 범주(글자)
    d[CAT_COLS] = d[CAT_COLS].fillna("None")                   # 범주형 빈 값 = 시설 없음 (설명서 정의)
    d[ZERO_COLS] = d[ZERO_COLS].fillna(0)                      # 시설 없음 → 면적·개수 0
    d["GarageYrBlt"] = d["GarageYrBlt"].fillna(d["YearBuilt"]) # 차고 없음 → 0 대신 집 건축 연도
    d["LotFrontage"] = d["LotFrontage"].fillna(lotfrontage_fill)  # 진짜 기록 누락 → 중앙값

    # ---- 2) 파생 변수: 사람이 집을 설명할 때 쓰는 숫자 ----
    d["TotalSF"] = d["TotalBsmtSF"] + d["1stFlrSF"] + d["2ndFlrSF"]       # 지하 + 1층 + 2층 총면적
    d["Age"] = d["YrSold"] - d["YearBuilt"]                               # 팔릴 때 집 나이
    d["RemodAge"] = d["YrSold"] - d["YearRemodAdd"]                       # 리모델링 후 지난 해
    d["TotalBath"] = (d["FullBath"] + 0.5 * d["HalfBath"]
                      + d["BsmtFullBath"] + 0.5 * d["BsmtHalfBath"])      # 욕실 합계 (반욕실 0.5)
    d["PorchSF"] = (d["OpenPorchSF"] + d["EnclosedPorch"] + d["3SsnPorch"]
                    + d["ScreenPorch"] + d["WoodDeckSF"])                 # 현관·데크 면적 합계

    # ---- 3) 품질 등급을 점수로 (원래 글자 열도 남겨 둠) ----
    for c in GRADE_COLS:
        d[c + "_n"] = d[c].map(GRADE).fillna(0)

    return d

def add_interactions(X):
    """상호작용 변수 7개 (이번 모델의 핵심). 원핫 인코딩이 끝난 피처 표의 맨 뒤에 붙인다."""
    X = X.copy()
    X["Qual_x_TotalSF"] = X["OverallQual"] * X["TotalSF"]       # 품질 × 총면적
    X["Qual_x_GrLiv"] = X["OverallQual"] * X["GrLivArea"]       # 품질 × 지상 거주 면적
    X["Qual2"] = X["OverallQual"] ** 2                          # 품질의 제곱
    X["Age_x_Qual"] = X["Age"] * X["OverallQual"]               # 연식 × 품질
    X["Garage_x_Qual"] = X["GarageCars"] * X["OverallQual"]     # 차고 크기 × 품질
    X["IsNew"] = (X["YearBuilt"] == X["YrSold"]).astype(float)  # 지은 해에 팔린 신축인가 (1/0)
    X["HasRemod"] = (X["YearRemodAdd"] != X["YearBuilt"]).astype(float)  # 리모델링 했는가 (1/0)
    return X

lf_median = train_df["LotFrontage"].median()     # 채우기 값은 train에서만 계산
train = prepare(train_df, lf_median)
test = prepare(test_df, lf_median)
print("전처리 후 train:", train.shape, "| test:", test.shape)
print("남은 빈 값 - train:", int(train.drop(columns="SalePrice").isna().sum().sum()),
      "| test:", int(test.isna().sum().sum()))

전처리 후 train: (1168, 96) | test: (292, 95)
남은 빈 값 - train: 0 | test: 0


`prepare` 함수로 train과 test에 **똑같은 전처리**를 적용하고, 상호작용 변수는 4단계에서 `add_interactions`로 붙인다. 아래와 같이 네 부분으로 나뉘게 된다.

1. **빈 값 처리:** 데이터 설명서에서 범주형 빈 값(NA)은 대부분 "해당 시설 없음"이라 `"None"`으로 채웁니다(최빈값으로 채우면 "수영장 없는 집"이 "보통 수영장 있는 집"이 됨). 지하실·차고 면적처럼 시설이 없으면 0인 숫자는 0으로, 도로 접면 길이처럼 진짜 누락은 train의 중앙값으로 채운다.
2. **파생 변수 5개:** 총면적, 매도 시 연식, 리모델링 후 경과 연수, 욕실 합계, 현관·데크 면적이다.
3. **품질 등급 점수 10개:** Ex/Gd/TA/Fa/Po를 5/4/3/2/1로 바꿔 "Gd가 TA보다 좋다"는 순서를 모델이 바로 알게 한다.
4. **상호작용 변수 7개 (`add_interactions`):** 집값은 "품질이 좋으면 + 넓으면"처럼 더하기가 아니라 **"고급 집은 넓어질수록 더 빨리 비싸진다"**처럼 곱하기로 움직이는 경우가 많다. 품질 × 면적, 품질의 제곱 같은 변수를 **직접** 만들어 주면 모델이 이런 관계를 쉽게 배우도록 한다. 이 영향으로 그레이디언트 부스팅의 교차검증 RMSE가 22,721 → 21,907로 크게 좋아진 것을 알 수 있다.

## 3. 이상치 확인

In [5]:
# 면적 4,000평방피트가 넘는데 30만 달러 미만에 팔린 집
special = ((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)).values
cols = ["Id", "Neighborhood", "GrLivArea", "OverallQual", "SalePrice", "SaleCondition", "SaleType", "YearBuilt", "YrSold"]
out = train.loc[special, cols].copy()
out["$/평방피트"] = (out["SalePrice"] / out["GrLivArea"]).round(1)
out

,Id,Neighborhood,GrLivArea,OverallQual,SalePrice,SaleCondition,SaleType,YearBuilt,YrSold,$/평방피트
64,1299,Edwards,5642,10,160000,Partial,New,2008,2008,28.4
373,524,Edwards,4676,10,184750,Partial,New,2007,2007,39.5


두 집(Id 524, 1299)은 같은 Edwards 동네의 품질 10등급 대형 신축인데, 평방피트당 28~40달러에 팔렸다. train 집들의 가운데 50%가 평방피트당 100~138달러인 것과 비교하면 정상 시장가로 보기 어려운 거래라고 할 수 있다. 이 2채가 학습에 들어가면 "넓은 집은 별로 안 비싸다"는 잘못된 경향을 배우므로 **학습에서만 제외**하기로 하고 기록한다.

## 4. 원핫 인코딩

In [6]:
FEATS = [c for c in train.columns if c not in ("Id", "SalePrice")]   # 모델에 넣을 열

# train과 test를 합쳐 한 번에 원핫 인코딩 → 두 데이터의 열 구성이 똑같아짐
X_all = pd.get_dummies(pd.concat([train[FEATS], test[FEATS]], keys=["train", "test"]), dtype=float)
X = add_interactions(X_all.loc["train"].reset_index(drop=True))       # 학습용 피처 + 상호작용 7개
X_test = add_interactions(X_all.loc["test"].reset_index(drop=True))   # 예측용 피처 + 상호작용 7개

y = train["SalePrice"].values      # 정답 (달러)
log_y = np.log(y)                  # 로그 변환한 정답 (모델은 이걸 학습)
print("X:", X.shape, "| X_test:", X_test.shape)

X: (1168, 339) | X_test: (292, 339)


범주형 열을 0/1 열로 바꾼다 (예: `Neighborhood_NAmes`). train과 test를 따로 변환하면 test에만 있는 범주 때문에 열 개수가 달라질 수 있어, **합친 뒤 한 번에** 변환하고 다시 나눈다. 원래 정보 79개 + 파생 5개 + 등급 점수 10개를 원핫 인코딩하면 332열이 되고, 그 뒤에 상호작용 변수 7개를 붙여 339열이 된다. (상호작용 변수를 맨 뒤에 붙이는 이유: 열 순서가 바뀌면 그레이디언트 부스팅의 무작위 선택이 달라져 결과가 조금씩 달라지기 때문에, 처음 실험과 같은 순서로 맞춘다.)

**로그 변환:** 정답 `y` 대신 `log(y)`를 학습하도록 한다. 집값은 금액보다 비율로 움직이기 때문에(10만 달러 집의 1만 달러와 50만 달러 집의 1만 달러는 의미가 다름), 로그로 바꾸면 모델이 "품질 1등급 = 가격 몇 % 상승"처럼 배워 고가·저가 모두에서 안정적이라고 할 수 있다. 예측한 뒤에는 `exp`로 다시 달러로 되돌리기로 한다.

## 5. 5-겹 교차검증 (80:20로 분리한 검증용 234채 한 번보다 1,168채 전체로 채점을 위해 5번 반복)
train을 5조각으로 나눠, 4조각으로 학습하고 남은 1조각으로 채점하기를 5번 반복한다. 모든 집이 한 번씩 "처음 보는 집"으로 채점되므로, 검증용 한 번보다 믿을 만한 점수를 얻을 수 있다.

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 각 조각에서 이상치 2채는 학습에서도, 채점에서도 뺌
#   (채점에서 빼는 이유: 정상 거래에 대한 실력을 재기 위해. 이 둘은 어떤 모델로도 100만 달러 가까이 빗나감)
folds = [(a[~special[a]], b[~special[b]]) for a, b in kf.split(X)]
scored = np.zeros(len(y), dtype=bool)
for _, b in folds:
    scored[b] = True

def rmse(true, pred):
    return np.sqrt(np.mean((true - pred) ** 2))

def out_of_fold(make_model):
    """5조각 교차검증 예측(OOF): 각 집은 자신이 빠진 4조각으로 학습한 모델의 예측을 받음.
    모델은 log(SalePrice)를 학습하고, 예측은 exp로 달러로 되돌림."""
    pred = np.zeros(len(y))
    for a, b in folds:
        model = make_model().fit(X.iloc[a], log_y[a])
        pred[b] = np.exp(model.predict(X.iloc[b]))
    return pred

`out_of_fold`는 모든 집에 대해 "자기 자신을 학습에 쓰지 않은 모델의 예측"을 만든다. 이 예측(OOF, out-of-fold)으로 RMSE를 계산하면 처음 보는 집에 대한 실력을 공정하게 잴 수 있고, 다음 단계에서 두 모델의 섞는 비율을 정할 때도 이 예측을 쓴다.

In [8]:
# 모델 A: 로그 Lasso — 표준화 후 Lasso(α=0.003)
make_lasso = lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.003, max_iter=200000))

# 모델 B: 그레이디언트 부스팅 — 로그 거래가 학습
make_gbr = lambda: GradientBoostingRegressor(
    loss="huber",          # 큰 오차에 덜 민감한 Huber 손실 (이상한 집에 휘둘리지 않게)
    n_estimators=2500,     # 트리 2,500개
    learning_rate=0.02,    # 트리 하나가 고치는 정도를 작게 → 천천히 꼼꼼히 학습
    max_depth=3,           # 트리 하나는 질문 3번까지 (단순한 트리를 많이)
    subsample=0.8,         # 트리마다 집의 80%만 무작위로 사용 → 과적합 방지
    max_features="sqrt",   # 질문마다 열의 일부만 후보로 → 다양한 트리
    min_samples_leaf=10,   # 잎 하나에 최소 10채 → 몇 채짜리 우연한 패턴 방지
    random_state=42)

t = time.time()
oof_lasso = out_of_fold(make_lasso)
print(f"A. 로그 Lasso            교차검증 RMSE {rmse(y[scored], oof_lasso[scored]):,.0f}  ({time.time()-t:.0f}초)")
t = time.time()
oof_gbr = out_of_fold(make_gbr)
print(f"B. 그레이디언트 부스팅   교차검증 RMSE {rmse(y[scored], oof_gbr[scored]):,.0f}  ({time.time()-t:.0f}초)")

A. 로그 Lasso            교차검증 RMSE 20,231  (1초)
B. 그레이디언트 부스팅   교차검증 RMSE 21,907  (61초)


성격이 다른 두 모델을 만들어 비교한다.

- **A. 로그 Lasso (α=0.003):** 모든 변수에 계수를 곱해 더하는 선형 모델이다. 변수를 표준화한 뒤 Lasso 벌점으로 쓸모없는 계수를 0으로 지운다. α는 교차검증에서 0.002~0.005를 비교해 가장 낮았던 값을 선택. 단독으로 교차검증 RMSE 약 20,200.
- **B. 그레이디언트 부스팅:** 작은 트리를 2,500개 쌓는 모델이다. 선형 모델이 못 잡는 "이 조건이면서 저 조건일 때" 같은 복잡한 관계를 잡는다. 단독으로는 약 21,900으로 Lasso보다 나쁘지만, **Lasso와 다른 곳에서 틀리기 때문에** 섞으면 서로의 실수를 상쇄한다.

## 6. 앙상블 비율 정하기

In [9]:
# 두 모델의 교차검증 예측을 열로 놓고, 실제 거래가에 가장 가깝게 만드는 '음수 없는 비율'을 계산
M = np.column_stack([oof_lasso, oof_gbr])
w, _ = nnls(M[scored], y[scored])
blend_oof = M @ w
print(f"비율: 로그 Lasso {w[0]:.3f} + 부스팅 {w[1]:.3f}  (합 {w.sum():.3f})")
print(f"앙상블 교차검증 RMSE: {rmse(y[scored], blend_oof[scored]):,.0f}")

# 중첩 검증: 비율을 4조각으로만 정하고, 남은 1조각으로 채점 (비율이 채점 데이터에 맞춰지는 것 방지)
fold_id = np.full(len(y), -1)
for i, (_, b) in enumerate(folds):
    fold_id[b] = i
rows = []
for i in range(5):
    fit_part, eval_part = (fold_id != i) & (fold_id >= 0), fold_id == i
    wi, _ = nnls(M[fit_part], y[fit_part])
    rows.append({"조각": i + 1,
                 "로그 Lasso 단독": round(rmse(y[eval_part], oof_lasso[eval_part])),
                 "부스팅 단독": round(rmse(y[eval_part], oof_gbr[eval_part])),
                 "앙상블 (중첩)": round(rmse(y[eval_part], M[eval_part] @ wi))})
nested = pd.DataFrame(rows).set_index("조각")
nested.loc["평균"] = nested.mean().round(0)
nested

비율: 로그 Lasso 0.672 + 부스팅 0.339  (합 1.010)
앙상블 교차검증 RMSE: 19,601


,로그 Lasso 단독,부스팅 단독,앙상블 (중첩)
조각,,,
1,17730.0,17961.0,17301.0
2,21783.0,22747.0,21312.0
3,21517.0,21026.0,20753.0
4,19150.0,21556.0,17617.0
5,20690.0,25575.0,21353.0
평균,20174.0,21773.0,19667.0


`nnls`(음수 없는 최소제곱)로 두 모델의 교차검증 예측을 어떤 비율로 섞어야 실제 거래가와 가장 가까운지 계산했다. 결과는 **로그 Lasso 0.672 + 부스팅 0.339**이고, 앙상블 교차검증 RMSE는 **19,601**로 두 모델 각각보다 낮습니다.

비율의 합이 1보다 조금 큰(약 1.01) 것은 의도된 결과입니다. 로그 예측을 `exp`로 달러로 되돌리면 평균적으로 살짝 낮게 나오는 편향이 있는데, 비율이 이를 자동으로 보정한 것임.

**중첩 검증:** 비율을 정할 때 쓴 데이터로 채점하면 점수가 낙관적이다. 그래서 비율은 4조각으로만 정하고 남은 1조각으로 채점하기를 5번 반복한 것임. 평균 **약 19,670**으로, 실제로 기대할 수 있는 실력은 이 정도입니다.

## 6-2. 템플릿 방식 검증: train.csv를 80:20으로 나누기
test.csv에는 정답이 없으므로, 템플릿처럼 **train.csv를 학습용 80%(934채)와 검증용 20%(234채)로 나눠** 처음 보는 집에 대한 점수를 확인하다. 검증용 234채는 학습에도, 앙상블 비율을 정하는 데도 전혀 쓰지 않는다.

In [10]:
# 큰 흐름은 ① 데이터 나누기 → ② 채점 도구 → ③ 앙상블 만들어 검증하기 → ④ 결과표 만들기

# 필요한 도구 불러오기
from sklearn.model_selection import train_test_split   # 데이터를 학습용/검증용으로 나누는 함수
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

# ① 데이터 나누기 (템플릿과 똑같은 코드)
X_tmpl = train_df.drop("SalePrice", axis=1)   # 정답 열(SalePrice)을 뺀 나머지 = 문제
y_tmpl = train_df["SalePrice"]                # 정답 열만 = 답
X_train, X_val, y_train, y_val = train_test_split(X_tmpl, y_tmpl, test_size=0.2, random_state=42)

# 우리가 실제로 모델에 넣을 X는 전처리(빈 값 처리, 파생 변수 등)를 마친 표라서 X_tmpl과 다름.
# 하지만 행 순서는 train_df와 똑같으므로, "몇 번째 행이 학습용/검증용인지" 번호만 꺼내서 X에 적용
tr_idx = X_train.index.values    # 학습용으로 뽑힌 행 번호 934개
va_idx = X_val.index.values      # 검증용으로 뽑힌 행 번호 234개

# special: 앞에서 찾은 이상치 2채 표시 (True/False 배열)
# ~special 은 "이상치가 아닌 것" → 학습용 중에서 이상치를 뺀 행 번호만 남김 (932개)
fit_idx = tr_idx[~special[tr_idx]]

# 확인용 출력: 학습 934 → 932, 검증 234, 검증 안에 이상치가 있는지(0이어야 함)
print("학습:", len(tr_idx), "→ 이상치 제외", len(fit_idx), "| 검증:", len(va_idx),
      "| 검증 안의 이상치:", int(special[va_idx].sum()))

학습: 934 → 이상치 제외 932 | 검증: 234 | 검증 안의 이상치: 0


In [11]:
# ② 채점 도구: 정답과 예측을 넣으면 네 가지 점수를 돌려주는 함수
def metrics(true, pred):
    return {
        "RMSE": round(rmse(true, pred)),                                   # 큰 오차에 벌점 더 주는 평균 오차 (주 지표)
        "MAE": round(mean_absolute_error(true, pred)),                     # 단순 평균 오차 (달러)
        "R²": round(r2_score(true, pred), 3),                              # 설명력
        "MAPE(%)": round(mean_absolute_percentage_error(true, pred) * 100, 1),  # % 오차
    }

In [12]:
# ③ 앙상블 모델을 만들고 검증용으로 예측하는 함수
#   (같은 과정을 '상호작용 변수 있는 표'와 '없는 표'에 두 번 쓰려고 함수로 만듦)
def holdout_ensemble(XX):
    """XX: 모델에 넣을 피처 표.
    학습용 932채만 써서
      1) 두 모델(로그 Lasso, 부스팅)을 몇 대 몇으로 섞을지 비율을 정하고
      2) 두 모델을 학습한 뒤
      3) 학습용과 검증용 각각을 예측해서 돌려준다.
    검증용 234채는 1), 2)에 전혀 쓰지 않으므로 '처음 보는 집'으로 채점된다."""

    # ---- 1) 섞는 비율 정하기 ----
    # 학습용 932채를 다시 5조각으로 나눔 (검증용은 여기 안 들어감)
    inner = KFold(n_splits=5, shuffle=True, random_state=42)

    # 932채 각각에 대해 [로그 Lasso 예측, 부스팅 예측] 두 칸을 담을 빈 표 (932행 × 2열, 처음엔 0)
    oof = np.zeros((len(fit_idx), 2))

    # 5조각을 돌아가며: 4조각으로 학습 → 남은 1조각 예측 (OOF 예측)
    for a, b in inner.split(fit_idx):
        ia, ib = fit_idx[a], fit_idx[b]          # a, b는 0~931 순번 → 실제 행 번호로 바꿈
        # 로그 Lasso: log(가격)을 학습 → 예측도 log 값 → np.exp로 달러로 되돌림
        oof[b, 0] = np.exp(make_lasso().fit(XX.iloc[ia], log_y[ia]).predict(XX.iloc[ib]))
        # 부스팅: 같은 방식
        oof[b, 1] = np.exp(make_gbr().fit(XX.iloc[ia], log_y[ia]).predict(XX.iloc[ib]))

    # nnls: "oof의 두 열에 각각 몇을 곱해 더하면 실제 가격(y)과 가장 가까울까?"를 계산
    #       곱하는 수(비율)는 음수가 안 나오게 제한. 결과 wh = [Lasso 비율, 부스팅 비율]
    wh, _ = nnls(oof, y[fit_idx])

    # ---- 2) 두 모델을 학습용 932채 전체로 학습 ----
    ml = make_lasso().fit(XX.iloc[fit_idx], log_y[fit_idx])   # 로그 Lasso
    mg = make_gbr().fit(XX.iloc[fit_idx], log_y[fit_idx])     # 부스팅

    # ---- 3) 학습용과 검증용을 각각 예측 ----
    out = {}
    for part, idx in [("학습", fit_idx), ("검증", va_idx)]:
        pl = np.exp(ml.predict(XX.iloc[idx]))    # 로그 Lasso 예측 (달러)
        pg = np.exp(mg.predict(XX.iloc[idx]))    # 부스팅 예측 (달러)
        out[part] = {
            "로그 Lasso": pl,
            "부스팅": pg,
            "앙상블": wh[0] * pl + wh[1] * pg,   # 1)에서 정한 비율로 섞은 예측
        }
    # out 예: out["검증"]["앙상블"] → 검증용 234채의 앙상블 예측
    return out, wh

In [13]:
# 실제 실행: 두 가지 피처 표로 비교
t = time.time()                                   # 걸린 시간 재기 시작

# (가) 이번 모델: 상호작용 변수 7개가 들어 있는 X
new, w_new = holdout_ensemble(X)

# (나) 비교용: 같은 X에서 상호작용 변수 7개만 뺀 표 → "상호작용 변수가 정말 도움이 되나?"를 확인
X_base = X.drop(columns=["Qual_x_TotalSF", "Qual_x_GrLiv", "Qual2", "Age_x_Qual",
                         "Garage_x_Qual", "IsNew", "HasRemod"])
old, w_old = holdout_ensemble(X_base)

print(f"완료 ({time.time()-t:.0f}초) | 이번 모델 비율: Lasso {w_new[0]:.3f} + 부스팅 {w_new[1]:.3f}")

완료 (150초) | 이번 모델 비율: Lasso 0.549 + 부스팅 0.460


In [14]:
# ④ 최종 결과표로 만들기
rows = {}   # {모델 이름: (학습 점수, 검증 점수)} 를 담을 상자

# 베이스라인: 모든 집에 학습용 평균 가격을 부르는 예측 (비교 기준선)
base_tr  = np.full(len(fit_idx), y[fit_idx].mean())   # 학습용 932채 모두 같은 값
base_val = np.full(len(va_idx),  y[fit_idx].mean())   # 검증용 234채 모두 같은 값
rows["베이스라인 (평균 예측)"] = (metrics(y[fit_idx], base_tr), metrics(y[va_idx], base_val))

# 상호작용 변수 없는 앙상블
rows["상호작용 변수 없음: 앙상블"] = (metrics(y[fit_idx], old["학습"]["앙상블"]),
                                    metrics(y[va_idx],  old["검증"]["앙상블"]))

# 이번 모델의 세 가지 예측 (로그 Lasso 단독, 부스팅 단독, 앙상블)
for k in ["로그 Lasso", "부스팅", "앙상블"]:
    rows[f"이번 모델: {k}"] = (metrics(y[fit_idx], new["학습"][k]),
                              metrics(y[va_idx],  new["검증"][k]))

# rows를 표(DataFrame)로 변환
#   각 모델의 학습 점수 4개 앞에 "학습 ", 검증 점수 4개 앞에 "검증 "을 붙여 한 줄로 펼침
#   예: {"학습 RMSE": 13571, ..., "검증 RMSE": 16885, ...}
#   .T 는 행과 열을 뒤집어서 "모델 = 행, 점수 = 열"이 되게 함
table = pd.DataFrame({
    name: {**{f"학습 {m}": v for m, v in tr.items()},
           **{f"검증 {m}": v for m, v in va.items()}}
    for name, (tr, va) in rows.items()
}).T
table

,학습 RMSE,학습 MAE,학습 R²,학습 MAPE(%),검증 RMSE,검증 MAE,검증 R²,검증 MAPE(%)
베이스라인 (평균 예측),80722.0,56929.0,0.000,36.7,71307.0,53797.0,-0.017,31.3
상호작용 변수 없음: 앙상블,14592.0,9514.0,0.967,5.8,17190.0,12119.0,0.941,7.1
이번 모델: 로그 Lasso,17613.0,11595.0,0.952,6.8,17730.0,12465.0,0.937,7.3
이번 모델: 부스팅,10442.0,6190.0,0.983,4.0,17265.0,12603.0,0.940,7.3
이번 모델: 앙상블,13571.0,8786.0,0.972,5.4,16885.0,12104.0,0.943,7.0


템플릿의 분할 코드(`train_test_split(X, y, test_size=0.2, random_state=42)`)를 그대로 쓰고, 그 분할이 고른 행 번호를 전처리된 피처에 적용했다. 이상치 2채는 모두 학습 쪽에 있어 학습은 932채, 검증은 234채 전체로 합니다. 앙상블 비율도 학습 부분 안의 5-겹 교차검증으로만 정해서, 검증용 234채는 점수를 매길 때 처음 본다.

| 모델 | 검증 RMSE | 검증 R² | 검증 MAPE | 학습 RMSE |
|---|---|---|---|---|
| 베이스라인 (평균 예측) | 71,307 | −0.017 | 31.3% | 80,722 |
| 상호작용 변수 없음: 앙상블 | 17,190 | 0.941 | 7.1% | 14,592 |
| 이번 모델: 로그 Lasso | 17,730 | 0.937 | 7.3% | 17,613 |
| 이번 모델: 부스팅 | 17,265 | 0.940 | 7.3% | 10,442 |
| **이번 모델: 앙상블** | **16,885** | **0.943** | **7.0%** | 13,571 |

- 이번 모델의 앙상블이 검증 RMSE **16,885**로 가장 좋고, 상호작용 변수가 없는 같은 구조(17,190)보다 305달러 낮다.
- 부스팅은 학습 RMSE(10,442)와 검증 RMSE(17,265)의 차이가 커서 학습 데이터를 어느 정도 외우는 경향이 있고, 로그 Lasso는 학습과 검증이 거의 같아 안정적이다. 둘을 섞으면 각각보다 낮아진다.
- 이 분할에서는 앙상블 비율이 0.549 : 0.460으로 train 전체로 정한 0.672 : 0.339와 다르다는 점을 유의해야 한다. 데이터가 적을수록 비율이 흔들리므로, 최종 제출에는 train 전체로 정한 비율을 쓰기로 한다.
- 참고로 5-겹 교차검증(5단계)의 첫 번째 조각이 이 검증용 234채와 정확히 같다 (같은 `random_state=42`의 무작위 순서를 쓰기 때문). 5-겹 교차검증 점수는 이 조각을 포함한 5조각의 평균이다.

## 7. 최종 학습과 제출 파일

In [15]:
# 퀘스트의 지시에 따라 템플릿처럼 '학습용 80%(X_train)'로 학습한 모델로 test를 예측
#   fit_idx: 6-2의 X_train 934채 중 이상치 2채를 뺀 932채
final_lasso = make_lasso().fit(X.iloc[fit_idx], log_y[fit_idx])
final_gbr = make_gbr().fit(X.iloc[fit_idx], log_y[fit_idx])

# test 예측: 각 모델의 로그 예측을 exp로 달러로 되돌린 뒤,
#   6-2에서 학습 부분 안의 교차검증으로 정한 비율(w_new)로 섞음
pred_lasso = np.exp(final_lasso.predict(X_test))
pred_gbr = np.exp(final_gbr.predict(X_test))
test_pred = w_new[0] * pred_lasso + w_new[1] * pred_gbr

print(f"학습한 집: {len(fit_idx)} (X_train 934채 - 이상치 2채) | 비율: Lasso {w_new[0]:.3f} + 부스팅 {w_new[1]:.3f}")
print("예측한 집:", len(test_pred), "| 빈 예측:", int(np.isnan(test_pred).sum()))
pd.Series(test_pred, name="예측 SalePrice").describe().round(0)

학습한 집: 932 (X_train 934채 - 이상치 2채) | 비율: Lasso 0.549 + 부스팅 0.460
예측한 집: 292 | 빈 예측: 0


count       292.0
mean     181186.0
std       76744.0
min       55682.0
25%      132115.0
50%      160442.0
75%      212751.0
max      612697.0
Name: 예측 SalePrice, dtype: float64

**설명:** 퀘스트 지시에 따라 템플릿과 같이 **학습용 80%(`X_train`)로 학습한 모델로 test를 예측**한다. 6-2에서 검증 RMSE 16,885를 낸 모델과 같은 설정이다(학습 932채, 비율 Lasso 0.549 + 부스팅 0.460). 따라서 이 제출 파일의 기대 성능은 검증 점수(RMSE 16,885)로 설명할 수 있다.

In [16]:
submission = pd.DataFrame({"Id": test_df["Id"], "SalePrice": test_pred})
submission.to_csv("submission_final.csv", index=False)
print(submission.shape)
submission.head()

(292, 2)


,Id,SalePrice
0,86,257083.442545
1,227,267354.550493
2,492,143157.705925
3,64,133422.412974
4,576,114214.260584


### 요약 정리
| 모델 | 교차검증 RMSE |
|---|---|
| A. 로그 Lasso (상호작용 변수 포함) | 약 20,230 · 80:20 검증 17,730 |
| B. 그레이디언트 부스팅 (상호작용 변수 포함) | 약 21,910 · 80:20 검증 17,265 |
| **A + B 앙상블** | **19,601** · **80:20 검증 16,885** |

제출 파일은 80:20 분할의 학습용(932채)으로 학습한 앙상블(Lasso 0.549 + 부스팅 0.460)로 만들었다.

**이 모델이 좋아진 이유 세 가지**
1. **로그 변환:** 집값을 비율로 학습해 고가·저가 모두 안정
2. **상호작용 변수:** "고급 집은 넓을수록 더 비싸진다" 같은 곱하기 관계를 직접 제공
3. **앙상블:** 선형 모델과 트리 모델이 서로 다른 곳에서 틀려 실수가 상쇄

**유의할 점**
- α, 트리 설정, 상호작용 변수는 train 전체의 교차검증 점수를 보며 골랐기 때문에, 검증 점수는 조금 낙관적일 수 있다.
- 로그 변환, 그레이디언트 부스팅, 교차검증은 수업 진도 밖 기법이나, 리더보드 스코어를 높이고 선행학습을 위해 클로드의 도움을 받아 시도했다. 단시간에 많은 것을 배울 수 있는 좋은 기회였다.
